# Demo 1: Weighted cycle repair with the paper-local governor

This notebook shows how the paper-local governor (`code/symbol_game_governor.py`) checks scripted proposals on a small finite graph. It uses a directed 3-cycle with penalties (1, 2, 2) and reports (1, 2, 2), whose quotient minimum is $\Phi_1 = 5$. This is a second triangle. The manuscript triangle uses different reports and weights, and its minimum is 3. A positive optimum returns HOLD. EXACT is reserved for cost zero. It demonstrates:

1. Asserting the three edge reports.
2. Refusal of a CUT that increases the proposal cost (`DefectIncrease`).
3. A CUT to a minimizing potential, which lowers the proposal cost to the unchanged minimum $\Phi_1 = 5$ and returns HOLD.

These are scripted fixtures demonstrating governor behavior. No live model call or comparative model-performance claim is made.

In [1]:
import contextlib
import io
import sys
from pathlib import Path

paper_dir = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / "code" / "certificate_gate.py").exists()), None)
if paper_dir is None:
    raise RuntimeError("Start the notebook from the paper directory or its jupyter_notebooks directory")
sys.path.insert(0, str(paper_dir / "code"))


@contextlib.contextmanager
def one_output_block():
    """Send a cell's printed lines as one stream message.

    Timed flushes could otherwise split output into several messages, and a
    stored notebook would then differ from a fresh run under load.
    """
    buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(buffer):
            yield
    finally:
        sys.stdout.write(buffer.getvalue())
        sys.stdout.flush()


from symbol_game_governor import ConstraintGraph, Edge, L1SymbolGameGovernor, MoveType

# Construct the 3-cycle carrier graph
vertices = ("a", "b", "c")
edges = (
    Edge("a", "b", weight=1),
    Edge("b", "c", weight=2),
    Edge("c", "a", weight=2),
)
graph = ConstraintGraph(vertices, edges)
with one_output_block():
    print(f"Carrier initialized with vertices {graph.vertices} and edges:")
    for e in graph.edges:
        print(f"  {e.u} -> {e.v} (weight {e.weight})")

Carrier initialized with vertices ('a', 'b', 'c') and edges:
  a -> b (weight 1)
  b -> c (weight 2)
  c -> a (weight 2)


## Step 1: Asserting the edge reports

The fixture asserts the edge values on the graph:

- $a \to b = 1$
- $b \to c = 2$
- $c \to a = 2$

In [2]:
gov = L1SymbolGameGovernor(graph)

with one_output_block():
    step1 = gov.execute_move(MoveType.ASSERT, {"u": "a", "v": "b", "value": 1})
    print(f"Turn 1: {step1.move_type.value} -> Defect={step1.defect}, Phi_1={step1.phi_1}")

    step2 = gov.execute_move(MoveType.ASSERT, {"u": "b", "v": "c", "value": 2})
    print(f"Turn 2: {step2.move_type.value} -> Defect={step2.defect}, Phi_1={step2.phi_1}")

    step3 = gov.execute_move(MoveType.ASSERT, {"u": "c", "v": "a", "value": 2})
    print(f"Turn 3: {step3.move_type.value} -> Defect={step3.defect}, Phi_1={step3.phi_1}")

Turn 1: ASSERT -> Defect=1, Phi_1=1
Turn 2: ASSERT -> Defect=5, Phi_1=3
Turn 3: ASSERT -> Defect=9, Phi_1=5


## Step 2: A cost-increasing CUT is refused

This scripted fixture submits a CUT that raises the proposal cost. The governor refuses it with `DefectIncrease` and leaves the accepted state unchanged.

In [3]:
# A scripted proposal whose potentials increase the cost
with one_output_block():
    bad_cut = gov.execute_move(MoveType.CUT, {"potentials": {"a": 0, "b": -5, "c": 10}})
    print(f"Verdict: {bad_cut.verdict.value}")
    print(f"Admitted: {bad_cut.admitted}")
    print(f"Governor Feedback:\n  {bad_cut.feedback}")

Verdict: REFUSAL
Admitted: False
Governor Feedback:
  Refused: DefectIncrease. Proposed defect 56 exceeds current defect 9.


## Step 3: Descent to the minimum

The paper-local reference builder supplies a minimizing potential. A CUT to that potential lowers the proposal cost $D(s)$ to $\Phi_1 = 5$; the CUT leaves $\Phi_1$ unchanged. Because that cost is positive, the verdict is HOLD. Each potential is sent with `str(v)`, so a later fractional value such as $-5/2$ is not truncated by `int`.

In [4]:
with one_output_block():
    phi1_opt, best_pot = gov.compute_phi_1()
    print(f"Optimal Phi_1: {phi1_opt}")
    print(f"Optimal Potential Assignment: {best_pot}")

    # Propose optimal potential assignment
    good_cut = gov.execute_move(MoveType.CUT, {"potentials": {k: str(v) for k, v in best_pot.items()}})
    print(f"Verdict: {good_cut.verdict.value}")
    print(f"Defect after CUT: {good_cut.defect}")
    print(f"Governor Feedback:\n  {good_cut.feedback}")

Optimal Phi_1: 5
Optimal Potential Assignment: {'a': Fraction(0, 1), 'c': Fraction(-2, 1), 'b': Fraction(-4, 1)}
Verdict: HOLD
Defect after CUT: 5
Governor Feedback:
  Admitted CUT. Section defect descended from 9 to 5. Phi_1([c])=5.
